# 3D Gaussian Splatting (3DGS) via Berkeley gsplat on Google Colab
### 3D Computer Vision Capstone Project

**Objective:** Train a continuous, differentiable 3D Gaussian radiance field on the registered 28 SfM keyframes of the smooth, specular bottle (`VID_20261102_015654418.mp4`).

**Theoretical Motivation:**
- Classical PatchMatch MVS assumes Lambertian (matte) reflectance, which struggles with dynamic light glare on shiny plastic/glass.
- 3D Gaussian Splatting models the scene as millions of anisotropic 3D Gaussians with **Spherical Harmonics (SH)** for view-dependent color rendering.
- Uses the exact camera poses ($R, t$) and initial sparse point cloud from your COLMAP Structure-from-Motion pipeline.

**Expected Runtime on Tesla T4:** $\approx 2-4$ minutes for 2,000 steps using `gsplat`.

### Step 1: Install gsplat & Clone Example Trainer

In [ ]:
%cd /content
# 1. Install gsplat and auxiliary dependencies
!pip install -q gsplat ninja jaxtyping rich viser tensorboard torchvision imageio tqdm tyro imageio-ffmpeg

# 2. Clone the gsplat repository for the simple_trainer
!rm -rf /content/gsplat_repo
!git clone https://github.com/nerfstudio-project/gsplat.git /content/gsplat_repo
!pip install -q -r /content/gsplat_repo/examples/requirements.txt

print("gsplat engine ready!")

### Step 2: Prepare Dataset Directory
Organizes your 28 registered keyframes and COLMAP camera poses (`sparse/0`) into standard 3DGS format.

In [ ]:
import os, shutil
from pathlib import Path

dataset_dir = Path("/content/3dgs_dataset")
images_dest = dataset_dir / "images"
sparse_dest = dataset_dir / "sparse" / "0"

images_dest.mkdir(parents=True, exist_ok=True)
sparse_dest.mkdir(parents=True, exist_ok=True)

# Copy existing keyframes and sparse COLMAP files
for img in Path("/content/keyframes").glob("*.jpg"):
    shutil.copy(img, images_dest / img.name)

for bin_file in Path("/content/sparse").glob("*.bin"):
    shutil.copy(bin_file, sparse_dest / bin_file.name)

print(f"Dataset prepared at {dataset_dir}:")
print(f"  Images: {len(list(images_dest.glob('*.jpg')))}")
print(f"  Sparse files: {[f.name for f in sparse_dest.glob('*.bin')]}")

### Step 3: Train 3D Gaussian Splatting (3DGS)
Trains 3D Gaussian Splats for 2,000 optimization steps with adaptive density control.

In [ ]:
%cd /content/gsplat_repo/examples

!python simple_trainer.py default \
    --data_dir /content/3dgs_dataset \
    --result_dir /content/3dgs_results \
    --max_steps 2000 \
    --save_ply \
    --disable_viewer

### Step 4: Package & Download Trained 3DGS Model
Finds the generated `.ply` Gaussian model and packages it into `3dgs_results.zip`.

In [ ]:
from IPython.display import FileLink
import zipfile
from pathlib import Path

results_dir = Path("/content/3dgs_results")
output_zip = Path("/content/3dgs_results.zip")

with zipfile.ZipFile(output_zip, "w", zipfile.ZIP_DEFLATED) as z:
    for ply_file in results_dir.rglob("*.ply"):
        z.write(ply_file, arcname=f"splats/{ply_file.name}")
        print(f"Added splat model: {ply_file.name} ({round(ply_file.stat().st_size / (1024*1024), 2)} MB)")
    for video in results_dir.rglob("*.mp4"):
        z.write(video, arcname=f"renders/{video.name}")
        print(f"Added video render: {video.name}")
    for img in results_dir.rglob("*.png"):
        z.write(img, arcname=f"renders/{img.name}")

print(f"\n3DGS Results packaged to {output_zip} ({round(output_zip.stat().st_size / (1024*1024), 2)} MB)")
FileLink('3dgs_results.zip')